# Notebook 3: End-to-End Pronunciation Scoring Demo
**Module 2 — PronounceAI: Acoustic Feature Analysis**

This notebook scores a user's pronunciation against a native reference using a **hybrid pipeline** that combines two independent scoring approaches.

---
## Why Two Scoring Approaches?

Pronunciation quality cannot be captured by a single number from a single method.
This notebook combines two complementary approaches:

### 1. Feature-Based Score (interpretable)
Directly compares the acoustic features of the user's audio to the reference audio.
- Computes **cosine similarity** for MFCC, Pitch (F0), and Energy (RMS).
- Tells you *how similar* the user's sound is to the reference on each acoustic dimension.
- **Strength:** Transparent and explainable — you can see which specific feature is off.
- **Limitation:** Purely mathematical. Does not know what "good pronunciation" looks like in general.

### 2. Model-Based Score (learned patterns)
Feeds the user's features into the **trained Random Forest and Neural Network** from Notebook 1.
- The models were trained on hundreds of good vs. degraded pronunciation examples.
- They output **P(good pronunciation)** — a probability learned from real data.
- **Strength:** Captures pronunciation patterns beyond surface similarity to one reference.
- **Limitation:** A black box; harder to explain *why* a score was given.

### Combined Final Score
```
Final Score = 0.6 × Feature Score  +  0.4 × Model Score
```
Feature scoring gets the higher weight (0.6) because it directly compares to the reference,
which is the most grounded signal in a pronunciation comparison task.
Model scoring adds robustness (0.4) by catching patterns the reference comparison might miss.

```
User Audio ──► extract_features() ──► MFCC sim  ─┐
Ref  Audio ──► extract_features() ──► Pitch sim ─┼─► Feature Score (0-100)
                                  ──► Energy sim─┘        × 0.6
                                                                  ╲
                                                                   ╲
                                                                    ╲──► Final Score
                                                                   ╱
                                                                  ╱
User Audio ──► extract_features() ──► scaler ──► RF  P(good) ─┐       × 0.4
                                             ──► NN  P(good) ─┴─► Model Score (0-100)
```
---

## Step 1: Install Dependencies

In [ ]:
!pip install librosa numpy pandas matplotlib soundfile scipy torch scikit-learn joblib ipywidgets

## Step 2: Imports and Configuration

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
import joblib
import torch
import torch.nn as nn
from pathlib import Path
from scipy.spatial.distance import cosine as cosine_dist

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
MODELS_DIR  = Path("../models")
SAMPLE_RATE = 16_000

# ── Hybrid scoring split ───────────────────────────────────────────────────
# Final Score = ALPHA * Feature_Score + (1-ALPHA) * Model_Score
ALPHA = 0.6   # weight for feature-based score

# ── Weights inside the Feature Score ──────────────────────────────────────
# These must sum to 1.0
FEATURE_WEIGHTS = {
    "mfcc"  : 0.50,   # strongest signal for articulation
    "pitch" : 0.30,   # intonation / tone
    "energy": 0.20,   # stress / rhythm
}

# ── Weights inside the Model Score ────────────────────────────────────────
# RF and NN are both trained on the same data; we average them equally
MODEL_WEIGHTS = {
    "rf": 0.50,
    "nn": 0.50,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Scoring formula: {ALPHA} × Feature Score + {1-ALPHA} × Model Score")

## Step 3: Neural Network Definition

> **Important:** This architecture must match exactly what was used in Notebook 1.
> Any change here will cause the saved weights to load incorrectly.

In [ ]:
class PronunciationNet(nn.Module):
    """
    3-layer MLP trained in Notebook 1.
    layer1 + layer2 = embedding; head = classifier.
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3))
        self.layer2 = nn.Sequential(nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.3))
        self.head   = nn.Linear(64, 2)

    def forward(self, x):
        return self.head(self.layer2(self.layer1(x)))

    def predict_proba_good(self, x: torch.Tensor) -> float:
        """Return P(good pronunciation) as a float in [0, 1]."""
        with torch.no_grad():
            logits = self.forward(x)
            proba  = torch.softmax(logits, dim=1)
        return float(proba[0, 1].cpu())

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        """Return 64-dim pronunciation embedding (for visualization only)."""
        with torch.no_grad():
            return self.layer2(self.layer1(x))

## Step 4: Load Models

We load the **scaler** (must be the same one used during training) and both models.

In [ ]:
# Load feature config saved by Notebook 1
with open(MODELS_DIR / "feature_config.json") as f:
    config = json.load(f)

N_MFCC    = config["n_mfcc"]      # must match Notebook 1
INPUT_DIM = config["input_dim"]   # must match Notebook 1
print(f"Feature config: N_MFCC={N_MFCC}, INPUT_DIM={INPUT_DIM}")

# Scaler — MUST be the same scaler fitted during training
scaler = joblib.load(MODELS_DIR / "scaler.joblib")
print("Scaler loaded.")

# Random Forest
rf_model = joblib.load(MODELS_DIR / "random_forest.joblib")
print("Random Forest loaded.")

# Neural Network
nn_model = PronunciationNet(INPUT_DIM).to(device)
nn_model.load_state_dict(
    torch.load(MODELS_DIR / "pronunciation_net.pt", map_location=device)
)
nn_model.eval()
print("Neural Network loaded.")

## Step 5: Feature Extraction

> **Consistency rule:** This function is identical to the one in Notebooks 1 and 2.
> Do NOT modify it — any change will produce features incompatible with the trained scaler.

In [ ]:
def load_audio(path: str) -> np.ndarray:
    """Load any audio file and resample to 16 kHz mono."""
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return y


def extract_features(y: np.ndarray) -> np.ndarray:
    """
    Extract the 86-dim feature vector used during training in Notebook 1.
    Layout: [mfcc_means(40), mfcc_stds(40), pitch_mean, pitch_std,
              rms_mean, rms_std, centroid_mean, centroid_std]
    """
    # MFCCs — capture vocal tract shape (pronunciation texture)
    mfcc      = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    mfcc_mean = np.mean(mfcc, axis=1)   # (N_MFCC,)
    mfcc_std  = np.std(mfcc,  axis=1)   # (N_MFCC,)

    # Pitch (F0) — captures intonation and tone
    f0, voiced_flag, _ = librosa.pyin(
        y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
    )
    f0_voiced  = f0[voiced_flag] if (voiced_flag is not None and voiced_flag.any()) else np.array([])
    pitch_mean = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
    pitch_std  = float(np.std(f0_voiced))  if len(f0_voiced) > 0 else 0.0

    # Energy (RMS) — captures loudness and stress patterns
    rms      = librosa.feature.rms(y=y)[0]

    # Spectral Centroid — captures brightness / clarity
    centroid = librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE)[0]

    return np.concatenate([
        mfcc_mean, mfcc_std,
        [pitch_mean, pitch_std,
         np.mean(rms), np.std(rms),
         np.mean(centroid), np.std(centroid)]
    ])


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors, clamped to [0, 1]."""
    return float(np.clip(1.0 - cosine_dist(a + 1e-9, b + 1e-9), 0.0, 1.0))


print("Feature extraction ready.")

## Step 6: Feature Score

Compares the user's audio directly to the reference audio using cosine similarity on three acoustic dimensions.

```
Feature Score = 0.50 × MFCC_similarity
              + 0.30 × Pitch_similarity
              + 0.20 × Energy_similarity
```

Each similarity is cosine similarity (0–1), then the weighted sum is scaled to 0–100.

In [ ]:
def compute_feature_score(y_user: np.ndarray, y_ref: np.ndarray) -> dict:
    """
    Compute the feature-based pronunciation score by comparing user and reference audio.

    Returns a dict with:
      - mfcc_sim    : MFCC cosine similarity (0-1)
      - pitch_sim   : Pitch cosine similarity (0-1)
      - energy_sim  : Energy cosine similarity (0-1)
      - feature_score : weighted combination scaled to 0-100
      - raw data    : f0 arrays and rms arrays for visualization
    """
    # ── MFCC similarity ────────────────────────────────────────────────────
    mfcc_user = np.mean(librosa.feature.mfcc(y=y_user, sr=SAMPLE_RATE, n_mfcc=N_MFCC), axis=1)
    mfcc_ref  = np.mean(librosa.feature.mfcc(y=y_ref,  sr=SAMPLE_RATE, n_mfcc=N_MFCC), axis=1)
    mfcc_sim  = cosine_sim(mfcc_user, mfcc_ref)

    # ── Pitch similarity ───────────────────────────────────────────────────
    f0_user, vf_user, _ = librosa.pyin(
        y_user, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
    )
    f0_ref, vf_ref, _ = librosa.pyin(
        y_ref,  fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7")
    )

    def pitch_summary(f0, vf):
        """Mean and std of F0 over voiced frames only."""
        f0v = f0[vf] if (vf is not None and vf.any()) else np.array([])
        return np.array([np.mean(f0v) if len(f0v) > 0 else 0.0,
                         np.std(f0v)  if len(f0v) > 0 else 0.0])

    pitch_sim = cosine_sim(pitch_summary(f0_user, vf_user),
                           pitch_summary(f0_ref,  vf_ref))

    # ── Energy similarity ──────────────────────────────────────────────────
    rms_user = librosa.feature.rms(y=y_user)[0]
    rms_ref  = librosa.feature.rms(y=y_ref)[0]
    min_len  = min(len(rms_user), len(rms_ref))
    energy_sim = cosine_sim(rms_user[:min_len], rms_ref[:min_len])

    # ── Weighted feature score (0–100) ─────────────────────────────────────
    feature_score = (
        FEATURE_WEIGHTS["mfcc"]   * mfcc_sim   +
        FEATURE_WEIGHTS["pitch"]  * pitch_sim  +
        FEATURE_WEIGHTS["energy"] * energy_sim
    ) * 100

    return {
        "mfcc_sim"      : round(mfcc_sim,     4),
        "pitch_sim"     : round(pitch_sim,    4),
        "energy_sim"    : round(energy_sim,   4),
        "feature_score" : round(feature_score, 2),
        # raw arrays needed for plots
        "_f0_user" : f0_user, "_vf_user": vf_user,
        "_f0_ref"  : f0_ref,  "_vf_ref" : vf_ref,
        "_rms_user": rms_user, "_rms_ref": rms_ref,
    }


print("compute_feature_score() ready.")

## Step 7: Model Score

Passes the user's features through the **trained Random Forest and Neural Network**.
Both models output P(good pronunciation). We average them to get the final Model Score.

```
Model Score = 0.50 × RF_P(good)  +  0.50 × NN_P(good)   (scaled to 0–100)
```

> The scaler must be applied **before** passing features to either model — the same scaler that was fitted in Notebook 1.

In [ ]:
def compute_model_score(y_user: np.ndarray) -> dict:
    """
    Predict pronunciation quality for the user's audio using both trained models.

    Returns a dict with:
      - rf_prob    : Random Forest P(good pronunciation) in [0, 1]
      - nn_prob    : Neural Network P(good pronunciation) in [0, 1]
      - model_score: weighted average scaled to 0–100
    """
    # Step 1: Extract the same 86-dim feature vector used at training time
    raw_features = extract_features(y_user)           # (86,) unscaled

    # Step 2: Apply the training scaler — critical for correct predictions
    scaled_features = scaler.transform([raw_features])  # (1, 86)

    # Step 3: Random Forest — P(good)
    rf_prob = float(rf_model.predict_proba(scaled_features)[0, 1])

    # Step 4: Neural Network — P(good)
    t = torch.tensor(scaled_features[0], dtype=torch.float32).unsqueeze(0).to(device)
    nn_prob = nn_model.predict_proba_good(t)

    # Step 5: Combine into a single model score (0–100)
    model_score = (
        MODEL_WEIGHTS["rf"] * rf_prob +
        MODEL_WEIGHTS["nn"] * nn_prob
    ) * 100

    return {
        "rf_prob"     : round(rf_prob,     4),
        "nn_prob"     : round(nn_prob,     4),
        "model_score" : round(model_score, 2),
    }


print("compute_model_score() ready.")

## Step 8: Feedback Generator

Generates human-readable feedback based on individual feature scores and the model score.

In [ ]:
def generate_feedback(feature_result: dict, model_result: dict) -> list:
    """
    Return a list of feedback strings based on thresholds.
    Each item is a (severity, message) tuple:
      severity = 'good' | 'warning' | 'issue'
    """
    feedback = []

    mfcc_sim   = feature_result["mfcc_sim"]
    pitch_sim  = feature_result["pitch_sim"]
    energy_sim = feature_result["energy_sim"]
    model_score = model_result["model_score"]

    # ── MFCC feedback (articulation / clarity) ─────────────────────────────
    if mfcc_sim < 0.60:
        feedback.append(("issue",   "Pronunciation clarity issue — vocal tract shape differs significantly from the reference"))
    elif mfcc_sim < 0.75:
        feedback.append(("warning", "Pronunciation clarity could improve — some articulation differences detected"))
    else:
        feedback.append(("good",    "Articulation clarity matches the reference well"))

    # ── Pitch feedback (intonation / tone) ─────────────────────────────────
    if pitch_sim < 0.60:
        feedback.append(("issue",   "Pitch/intonation needs improvement — tone pattern deviates from the reference"))
    elif pitch_sim < 0.75:
        feedback.append(("warning", "Pitch/intonation is close but not yet natural — focus on tone variation"))
    else:
        feedback.append(("good",    "Intonation and pitch match the reference"))

    # ── Energy feedback (volume / stress) ──────────────────────────────────
    if energy_sim < 0.60:
        feedback.append(("issue",   "Speaking volume inconsistency — energy/stress pattern differs from the reference"))
    elif energy_sim < 0.75:
        feedback.append(("warning", "Minor speaking volume inconsistency — stress timing could be improved"))
    else:
        feedback.append(("good",    "Speaking volume and stress pattern match the reference"))

    # ── Overall model-based quality ────────────────────────────────────────
    if model_score < 50:
        feedback.append(("issue",   f"Overall speech quality is low ({model_score:.0f}/100) — the model detects significant pronunciation issues"))
    elif model_score < 70:
        feedback.append(("warning", f"Overall speech quality is moderate ({model_score:.0f}/100) — some patterns differ from native speech"))
    else:
        feedback.append(("good",    f"Overall speech quality is good ({model_score:.0f}/100) — the model rates this as natural pronunciation"))

    return feedback


print("generate_feedback() ready.")

## Step 9: Main Scoring Pipeline

Calls both scoring functions and combines them with the documented formula.

In [ ]:
def score_pronunciation(user_path: str, ref_path: str, verbose: bool = True) -> dict:
    """
    Full hybrid pronunciation scoring pipeline.

    Parameters
    ----------
    user_path : path to the user's audio recording
    ref_path  : path to the reference (native) audio
    verbose   : if True, print a formatted score report

    Returns
    -------
    dict with keys:
      feature_score, model_score, final_score,
      mfcc_sim, pitch_sim, energy_sim,
      rf_prob, nn_prob, feedback,
      and raw audio arrays for plotting
    """
    # ── Load audio ────────────────────────────────────────────────────────
    y_user = load_audio(user_path)
    y_ref  = load_audio(ref_path)

    # ── Track 1: Feature-based score ──────────────────────────────────────
    feat = compute_feature_score(y_user, y_ref)

    # ── Track 2: Model-based score ────────────────────────────────────────
    mdl = compute_model_score(y_user)

    # ── Combine: Final Score = 0.6 × Feature + 0.4 × Model ───────────────
    final_score = ALPHA * feat["feature_score"] + (1 - ALPHA) * mdl["model_score"]

    # ── Generate feedback from both tracks ────────────────────────────────
    feedback = generate_feedback(feat, mdl)

    result = {
        # scores
        "feature_score" : feat["feature_score"],
        "model_score"   : mdl["model_score"],
        "final_score"   : round(final_score, 2),
        # feature track breakdown
        "mfcc_sim"      : feat["mfcc_sim"],
        "pitch_sim"     : feat["pitch_sim"],
        "energy_sim"    : feat["energy_sim"],
        # model track breakdown
        "rf_prob"       : mdl["rf_prob"],
        "nn_prob"       : mdl["nn_prob"],
        # feedback
        "feedback"      : feedback,
        # raw data for plots
        "_y_user"   : y_user,           "_y_ref"   : y_ref,
        "_f0_user"  : feat["_f0_user"], "_vf_user" : feat["_vf_user"],
        "_f0_ref"   : feat["_f0_ref"],  "_vf_ref"  : feat["_vf_ref"],
        "_rms_user" : feat["_rms_user"],"_rms_ref" : feat["_rms_ref"],
    }

    if verbose:
        icons = {"good": "✓", "warning": "~", "issue": "!"}
        print("\n" + "═" * 62)
        print(f"   FINAL PRONUNCIATION SCORE :  {final_score:.1f} / 100")
        print("═" * 62)
        print(f"   Feature Score  (weight {ALPHA:.0%}) :  {feat['feature_score']:.1f} / 100")
        print(f"     MFCC similarity    (50%) :  {feat['mfcc_sim']*100:.1f}%")
        print(f"     Pitch similarity   (30%) :  {feat['pitch_sim']*100:.1f}%")
        print(f"     Energy similarity  (20%) :  {feat['energy_sim']*100:.1f}%")
        print("─" * 62)
        print(f"   Model Score   (weight {1-ALPHA:.0%}) :  {mdl['model_score']:.1f} / 100")
        print(f"     Random Forest P(good)    :  {mdl['rf_prob']*100:.1f}%")
        print(f"     Neural Network P(good)   :  {mdl['nn_prob']*100:.1f}%")
        print("─" * 62)
        print("   FEEDBACK:")
        for severity, msg in feedback:
            print(f"     {icons[severity]}  {msg}")
        print("═" * 62)

    return result


print("score_pronunciation() ready.")

## Step 10: Visualization — MFCC Heatmaps, Pitch, Energy

In [ ]:
def plot_acoustic_comparison(result: dict, save_path: str = None):
    """3-row plot: MFCC side-by-side, pitch curves, energy curves."""
    y_user = result["_y_user"]
    y_ref  = result["_y_ref"]
    sr     = SAMPLE_RATE

    fig = plt.figure(figsize=(16, 13))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.3)

    # ── Row 1: MFCC heatmaps ───────────────────────────────────────────────
    for col, (y, label, sim_label) in enumerate([
        (y_user, "User Audio",      ""),
        (y_ref,  "Reference Audio", f"MFCC sim: {result['mfcc_sim']*100:.1f}%"),
    ]):
        ax  = fig.add_subplot(gs[0, col])
        mfc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        img = librosa.display.specshow(mfc, sr=sr, x_axis="time", ax=ax, cmap="viridis")
        ax.set_title(f"MFCC — {label}  {sim_label}")
        fig.colorbar(img, ax=ax)

    # ── Row 2: Pitch curves ────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(librosa.times_like(result["_f0_user"], sr=sr),
             result["_f0_user"], label="User",      color="steelblue", linewidth=1.5)
    ax2.plot(librosa.times_like(result["_f0_ref"],  sr=sr),
             result["_f0_ref"],  label="Reference", color="coral",     linewidth=1.5, alpha=0.8)
    ax2.set_title(f"Pitch (F0) — Pitch Similarity: {result['pitch_sim']*100:.1f}%")
    ax2.set_xlabel("Time (s)"); ax2.set_ylabel("Frequency (Hz)")
    ax2.legend()

    # ── Row 3: Energy curves ───────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2, :])
    ax3.plot(result["_rms_user"], label="User",      color="steelblue", linewidth=1.2)
    ax3.plot(result["_rms_ref"],  label="Reference", color="coral",     linewidth=1.2, alpha=0.8)
    ax3.set_title(f"Energy (RMS) — Energy Similarity: {result['energy_sim']*100:.1f}%")
    ax3.set_xlabel("Frame"); ax3.set_ylabel("RMS Energy")
    ax3.legend()

    fig.suptitle(
        f"PronounceAI Module 2 — Acoustic Comparison\n"
        f"Feature Score: {result['feature_score']:.1f}  |  "
        f"Model Score: {result['model_score']:.1f}  |  "
        f"Final Score: {result['final_score']:.1f} / 100",
        fontsize=13, fontweight="bold", y=1.01
    )

    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


print("plot_acoustic_comparison() ready.")

## Step 11: Score Breakdown Chart

Shows both tracks side-by-side and how they combine into the final score.

In [ ]:
def plot_score_breakdown(result: dict, save_path: str = None):
    """
    3-panel chart:
      Left   — Feature Score sub-components (MFCC, Pitch, Energy)
      Centre — Model Score sub-components (RF, NN)
      Right  — Final Score composition (Feature × 0.6 + Model × 0.4)
    """
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # ── Left: Feature Score sub-components ────────────────────────────────
    feat_labels = ["MFCC\n(50%)", "Pitch\n(30%)", "Energy\n(20%)"]
    feat_vals   = [
        result["mfcc_sim"]   * 100,
        result["pitch_sim"]  * 100,
        result["energy_sim"] * 100,
    ]
    feat_colors = ["#4C9BE8", "#E88A4C", "#4CE88F"]

    bars = axes[0].bar(feat_labels, feat_vals, color=feat_colors, edgecolor="white", width=0.5)
    axes[0].set_ylim(0, 115)
    axes[0].axhline(70, color="red",   linestyle="--", linewidth=1.2, label="70 threshold")
    axes[0].axhline(90, color="green", linestyle="--", linewidth=1.2, label="90 excellent")
    for bar, val in zip(bars, feat_vals):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                     f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")
    axes[0].set_title(f"Feature Score: {result['feature_score']:.1f}/100", fontsize=12)
    axes[0].set_ylabel("Similarity (%)")
    axes[0].legend(fontsize=8)

    # ── Centre: Model Score sub-components ────────────────────────────────
    mdl_labels = ["Random Forest\nP(good)", "Neural Network\nP(good)"]
    mdl_vals   = [result["rf_prob"] * 100, result["nn_prob"] * 100]
    mdl_colors = ["#9B4CE8", "#E84C9B"]

    bars2 = axes[1].bar(mdl_labels, mdl_vals, color=mdl_colors, edgecolor="white", width=0.5)
    axes[1].set_ylim(0, 115)
    axes[1].axhline(70, color="red",   linestyle="--", linewidth=1.2)
    axes[1].axhline(90, color="green", linestyle="--", linewidth=1.2)
    for bar, val in zip(bars2, mdl_vals):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                     f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")
    axes[1].set_title(f"Model Score: {result['model_score']:.1f}/100", fontsize=12)
    axes[1].set_ylabel("P(good) × 100")

    # ── Right: Final Score composition ────────────────────────────────────
    final_labels      = ["Feature\n(×0.6)", "Model\n(×0.4)", "FINAL\nSCORE"]
    final_vals        = [
        result["feature_score"] * ALPHA,
        result["model_score"]   * (1 - ALPHA),
        result["final_score"],
    ]
    final_colors = ["#4C9BE8", "#9B4CE8", "#E8A04C"]

    bars3 = axes[2].bar(final_labels, final_vals, color=final_colors, edgecolor="white", width=0.5)
    axes[2].set_ylim(0, 115)
    axes[2].axhline(70, color="red",   linestyle="--", linewidth=1.2, label="70 threshold")
    axes[2].axhline(90, color="green", linestyle="--", linewidth=1.2, label="90 excellent")
    for bar, val in zip(bars3, final_vals):
        axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                     f"{val:.1f}", ha="center", fontsize=11, fontweight="bold")
    axes[2].set_title(
        f"Final Score: {result['final_score']:.1f}/100\n"
        f"= 0.6×{result['feature_score']:.0f} + 0.4×{result['model_score']:.0f}",
        fontsize=12
    )
    axes[2].set_ylabel("Points")
    axes[2].legend(fontsize=8)

    plt.suptitle("Pronunciation Score Breakdown — Hybrid Pipeline",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


print("plot_score_breakdown() ready.")

## Step 12: Feedback Display

In [ ]:
def print_feedback(result: dict):
    """Pretty-print feedback with color-coded severity markers."""
    icons = {"good": "✓", "warning": "~", "issue": "!"}
    labels = {"good": "GOOD   ", "warning": "NOTE   ", "issue": "IMPROVE"}

    print("\n── Feedback " + "─" * 48)
    for severity, msg in result["feedback"]:
        print(f"  [{labels[severity]}] {icons[severity]}  {msg}")
    print("─" * 60)

    # Score interpretation
    s = result["final_score"]
    if s >= 90:
        level = "Excellent — near-native pronunciation"
    elif s >= 70:
        level = "Good — minor improvements needed"
    elif s >= 50:
        level = "Fair — noticeable pronunciation issues"
    else:
        level = "Needs practice — significant mismatch with reference"
    print(f"  Overall: {s:.1f}/100 → {level}")
    print("─" * 60)

## Step 13: Run the Demo

Set `USER_AUDIO` and `REF_AUDIO` to your audio file paths.
Supported formats: `.flac`, `.wav`, `.mp3`.

In [ ]:
LIBRI_TEST    = Path("../../../librispeech/LibriSpeech/test-clean")
all_test_flac = sorted(LIBRI_TEST.rglob("*.flac"))

# ── Edit these two paths to use your own recordings ────────────────────────
USER_AUDIO = str(all_test_flac[0])   # replace with your recording
REF_AUDIO  = str(all_test_flac[1])   # replace with the native reference

print(f"User audio : {Path(USER_AUDIO).name}")
print(f"Reference  : {Path(REF_AUDIO).name}")

In [ ]:
# Run the full hybrid pipeline
result = score_pronunciation(USER_AUDIO, REF_AUDIO)

# Print structured feedback
print_feedback(result)

# Score breakdown chart (3 panels)
plot_score_breakdown(result, save_path=str(MODELS_DIR / "score_breakdown.png"))

# Acoustic visualizations (MFCC, pitch, energy)
plot_acoustic_comparison(result, save_path=str(MODELS_DIR / "demo_comparison.png"))

## Step 14: Interactive File Upload (Jupyter)

Upload any two audio files and run the full pipeline.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import tempfile

upload_user = widgets.FileUpload(accept="audio/*", multiple=False, description="User Audio")
upload_ref  = widgets.FileUpload(accept="audio/*", multiple=False, description="Reference Audio")
run_btn     = widgets.Button(description="Score Pronunciation", button_style="success")
output_box  = widgets.Output()

def on_run_clicked(_):
    with output_box:
        output_box.clear_output()
        if not upload_user.value or not upload_ref.value:
            print("Please upload both audio files first.")
            return

        def save_upload(uploader):
            file_data = list(uploader.value.values())[0]
            tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
            tmp.write(file_data["content"])
            tmp.flush()
            return tmp.name

        res = score_pronunciation(save_upload(upload_user), save_upload(upload_ref))
        print_feedback(res)
        plot_score_breakdown(res)
        plot_acoustic_comparison(res)

run_btn.on_click(on_run_clicked)
display(widgets.VBox([
    widgets.HBox([upload_user, upload_ref]),
    run_btn,
    output_box
]))

## Step 15: Batch Scoring

Score multiple user recordings against one reference. Shows both Feature Score and Model Score for each file.

In [ ]:
def batch_score(user_files: list, ref_file: str) -> pd.DataFrame:
    """
    Score multiple user recordings against one reference.
    Returns a DataFrame sorted by Final Score (descending).
    """
    rows = []
    for path in user_files:
        try:
            r = score_pronunciation(path, ref_file, verbose=False)
            # Collect only the worst feedback item for the summary column
            issues  = [msg for sev, msg in r["feedback"] if sev == "issue"]
            summary = issues[0] if issues else r["feedback"][0][1]
            rows.append({
                "File"           : Path(path).name,
                "Feature Score"  : r["feature_score"],
                "  MFCC sim %"   : round(r["mfcc_sim"]   * 100, 1),
                "  Pitch sim %"  : round(r["pitch_sim"]  * 100, 1),
                "  Energy sim %" : round(r["energy_sim"] * 100, 1),
                "Model Score"    : r["model_score"],
                "  RF P(good) %" : round(r["rf_prob"]    * 100, 1),
                "  NN P(good) %" : round(r["nn_prob"]    * 100, 1),
                "Final Score"    : r["final_score"],
                "Top Issue"      : summary[:60],
            })
        except Exception as e:
            rows.append({
                "File": Path(path).name,
                "Final Score": None,
                "Top Issue": str(e)
            })

    return pd.DataFrame(rows).sort_values("Final Score", ascending=False)


# Example: score 6 files against the first reference
user_samples = [str(p) for p in all_test_flac[2:8]]
reference    = str(all_test_flac[0])

batch_df = batch_score(user_samples, reference)
pd.set_option("display.max_colwidth", 60)
display(batch_df)

---
## Module 2 Demo Complete

### Hybrid Pipeline Summary

| Track | Input | What it uses | What it measures |
|---|---|---|---|
| **Feature Score** (60%) | User + Reference audio | Cosine similarity of MFCC, Pitch, Energy | Acoustic distance to reference |
| **Model Score** (40%) | User audio only | RF + NN trained classifier | Absolute pronunciation quality |
| **Final Score** | — | 0.6 × Feature + 0.4 × Model | Overall pronunciation quality |

### Score Thresholds

| Score | Meaning |
|---|---|
| 90–100 | Excellent — near-native pronunciation |
| 70–89  | Good — minor improvements needed |
| 50–69  | Fair — noticeable pronunciation issues |
| < 50   | Needs practice — significant mismatch |